In [4]:
import mlrun

from dotenv import load_dotenv
# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
load_dotenv() 

import os
# https://docs.mlrun.org/en/stable/store/datastore.html#s3
# print(os.environ['AWS_ACCESS_KEY_ID'])
# print(os.environ['AWS_SECRET_ACCESS_KEY'])
# print(os.environ['MLRUN_AWS_ROLE_ARN'])

from pathlib import Path

artifact_path = Path.cwd().parent / "mlrun-data/"
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
print(artifact_path)
p = mlrun.set_environment(api_path="http://localhost:8080", artifact_path=artifact_path)

file://c:/Work/Folder_1/Project_folder/LLM_project_3_FineTune_MLOps/Finetune-legal-llm-mlops/mlrun-data


In [5]:
project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory

# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

In [6]:
pipeline_run = project.run(
    name="register_raw_datasets",
    arguments={
        "source_path": "s3://legal-llama-data/raw",
        "version": "1.0.0"
    },
    local=True,   # Run the pipeline sequence locally
    watch=True    # Print the progress to the console
)

> 2026-04-01 11:24:22,969 [warning] WARNING!, You seem to have uncommitted git changes, use .push()


> 2026-04-01 11:24:23,102 [info] Storing function: {'name': 'raw-data-register-function-register-process-raw-data', 'uid': '8c25de7b61534ee5a0913d85150781d3', 'db': None}
s3://legal-llama-data/raw/train.parquet processed and written to s3://legal-llama-data/processed_training/1.0.0


project,uid,iter,start,state,name,labels,inputs,parameters,results,artifacts
finetune-legal-extractor,...150781d3,0,Apr 01 03:24:23,completed,raw-data-register-function-register-process-raw-data,workflow=f394c747c6354144a73d6e0f3e6d2e17kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=train_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,,train_data


> 2026-04-01 11:24:36,625 [info] Run execution finished: {'status': 'completed', 'name': 'raw-data-register-function-register-process-raw-data'}
> 2026-04-01 11:24:36,630 [info] Storing function: {'name': 'raw-data-register-function-register-process-raw-data', 'uid': '598cfd351e424707816af4be1f91ddae', 'db': None}
s3://legal-llama-data/raw/validation.parquet processed and written to s3://legal-llama-data/processed_training/1.0.0


project,uid,iter,start,state,name,labels,inputs,parameters,results,artifacts
finetune-legal-extractor,...1f91ddae,0,Apr 01 03:24:36,completed,raw-data-register-function-register-process-raw-data,workflow=f394c747c6354144a73d6e0f3e6d2e17kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=validation_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,,validation_data


> 2026-04-01 11:24:40,282 [info] Run execution finished: {'status': 'completed', 'name': 'raw-data-register-function-register-process-raw-data'}
> 2026-04-01 11:24:40,284 [info] Storing function: {'name': 'raw-data-register-function-register-process-raw-data', 'uid': '90bb4baea2214bc6ae131fd36ff04bec', 'db': None}
s3://legal-llama-data/raw/test.parquet processed and written to s3://legal-llama-data/processed_training/1.0.0


project,uid,iter,start,state,name,labels,inputs,parameters,results,artifacts
finetune-legal-extractor,...6ff04bec,0,Apr 01 03:24:40,completed,raw-data-register-function-register-process-raw-data,workflow=f394c747c6354144a73d6e0f3e6d2e17kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=test_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,,test_data


> 2026-04-01 11:24:45,232 [info] Run execution finished: {'status': 'completed', 'name': 'raw-data-register-function-register-process-raw-data'}


uid,start,state,name,parameters,results
...150781d3,Apr 01 03:24:23,completed,raw-data-register-function-register-process-raw-data,label_column=inferenceartifact_key=train_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,
...1f91ddae,Apr 01 03:24:36,completed,raw-data-register-function-register-process-raw-data,label_column=inferenceartifact_key=validation_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,
...6ff04bec,Apr 01 03:24:40,completed,raw-data-register-function-register-process-raw-data,label_column=inferenceartifact_key=test_dataversion=1.0.0output_uri_path=s3://legal-llama-data/processed_training,


> 2026-04-01 11:24:45,243 [info] Started run workflow finetune-legal-extractor-register_raw_datasets with run id = 'f394c747c6354144a73d6e0f3e6d2e17' by local engine


In [ ]:
## Testing direct access to S3 data
data_uri = "s3://legal-llama-data/raw/test.parquet"

# Fetch the item and immediately convert it to a Pandas DataFrame
df = mlrun.get_dataitem(data_uri).as_df()
df.head()

In [7]:
## Testing data versioning and access to registered datasets in MLRun
data_uri = "store://datasets/finetune-legal-extractor/raw-data-register-function-register-process-raw-data_test_data:latest"
data_uri = "store://datasets/finetune-legal-extractor/raw-data-register-function-register-process-raw-data_test_data:1.0.0"

# Fetch the item and immediately convert it to a Pandas DataFrame
df = mlrun.get_dataitem(data_uri).as_df()
df.head()

,document_id,text,inference,timestamp,origin
0,1,NON-DISCLOSURE AGREEMENT\nRequired under JEA's...,[{'hypothesis': 'Receiving Party shall destroy...,2026-04-01 11:24:42.584790,downloaded
1,2,MUTUAL NON-DISCLOSURE AGREEMENT\nBetween\nAND\...,"[{'hypothesis': ""Receiving Party shall not rev...",2026-04-01 11:24:42.584790,downloaded
2,4,Non-Disclosure Agreement\nDate:\nParties: [NAM...,[{'hypothesis': 'Agreement shall not grant Rec...,2026-04-01 11:24:42.584790,downloaded
3,5,Confidentiality Agreement\nThis Confidentialit...,[{'hypothesis': 'Receiving Party shall not dis...,2026-04-01 11:24:42.584790,downloaded
4,6,MUTUAL NON-DISCLOSURE/CONFIDENTIALITY AGREEMEN...,[{'hypothesis': 'Receiving Party shall destroy...,2026-04-01 11:24:42.584790,downloaded


In [3]:
# This deletes all artifacts in the database
db = mlrun.get_run_db()
# Wipe all artifacts (including datasets) from this specific project
db.del_artifacts(project=project.metadata.name)

In [12]:
project.spec.get_code_path()

'../'